# Ⅱ-05 · 모델 만들고 점수 매기기

교과서 **104~112쪽**을 코랩에서 그대로 해 봅니다.
[Ⅱ단원 수업 자료](https://richee-pc.github.io/AI_cs/unit2.html#algo)에서 손으로 움직여 본 **선형 회귀**와 **k-최근접 이웃**을, 이번에는 진짜 데이터로 만들어 봅니다.

---

### 시작하기 전에

1. 맨 위 **파일 → 드라이브에 사본 저장** 을 누르세요.
   사본을 만들지 않으면 **내가 고친 것이 저장되지 않습니다.**
2. 셀 왼쪽 **▶** 를 누르거나, 셀 안을 클릭하고 **Shift + Enter**.
3. **반드시 위에서부터 차례로** 실행하세요.

> ### `____` 를 채우세요
> 코드에 **`____`** 가 보이면 거기가 오늘의 문제입니다.
> **빈칸 바로 위 줄에 힌트**가 있습니다. 채우지 않고 실행하면 빨간 글씨가 나는데,
> **고장이 아니라 «아직 안 채웠다»는 뜻**입니다.


## 오늘 하는 일

모델을 만드는 순서는 **어떤 모델이든 똑같습니다.**

| | 하는 일 | 코드 |
|---|---|---|
| ① | 연장 가져오기 | `import` |
| ② | 데이터 읽기 | `pd.read_csv(...)` |
| ③ | **속성 x** 와 **정답 y** 나누기 | `x = df[[...]]` · `y = df[...]` |
| ④ | **훈련용·테스트용** 나누기 | `train_test_split(...)` |
| ⑤ | 모델 만들고 **학습** | `model.fit(x_train, y_train)` |
| ⑥ | **맞혀 보고 점수** | `model.predict(...)` · `score(...)` |

앞부분에서 **숫자를 맞히는 예측 모델**을, 뒷부분에서 **무엇인지 가려내는 분류 모델**을 만듭니다.
**⑤번 한 줄만 바뀝니다.** 나머지는 글자 하나 안 바뀝니다 — 그것을 눈으로 확인하는 것이 오늘의 목표입니다.

---
# 앞부분 · 예측 모델

## 도서관과 공연장 수로 청년 인구를 맞혀 봅시다

도시를 새로 만들 때, **기반 시설이 많으면 사람이 모입니다.**
그렇다면 도서관 수와 공연장 수만 알면 그 지역 청년 인구를 짐작할 수 있을까요?

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
df = pd.read_csv(주소 + 'youth_population.csv')

df.head()

`head()` 는 **위에서 다섯 줄만** 보여 줍니다. 250줄을 다 띄우면 화면이 넘치니까요.

표 전체가 어떻게 생겼는지는 `info()` 로 봅니다.

In [ ]:
df.info()

**250 entries**(250줄), **5 columns**(5열), 전부 `int64`(정수)입니다.
`non-null` 이 전부 250이면 **빈칸이 하나도 없다**는 뜻입니다 — 오늘은 전처리할 게 없네요.

---
## ③ 속성(x)과 정답(y) 나누기

- **x** = 맞히는 데 **쓰는** 것 → 도서관 수, 공연장 수
- **y** = **맞히려는** 것 → 청년 인구

> **힌트** · 열 이름을 따옴표에 넣어 그대로 적으면 됩니다. 대괄호가 **두 겹**인 것을 눈여겨보세요.

In [ ]:
x = df[[____, ____]]
y = df[['청년']]

x.shape, y.shape

`((250, 2), (250, 1))` 이 나왔으면 맞습니다. 250줄, x 는 2열, y 는 1열.

> **대괄호 한 겹과 두 겹**
> `df['청년']` → 한 줄짜리(시리즈) · `df[['청년']]` → **표**(데이터프레임).
> 헷갈리면 **«여러 개 고를 거니까 두 겹»** 으로 외우세요.

---
## ④ 훈련용과 테스트용 나누기

**공부한 문제만 잘 푸는 건 실력이 아닙니다.**
그래서 데이터 일부를 **떼어 두었다가** 시험지로 씁니다.

- `test_size=0.2` → 전체의 **20%** 를 시험지로
- `random_state=0` → 섞는 방식을 **고정**. 이게 없으면 돌릴 때마다 결과가 달라집니다

> **힌트** · 전체의 20% 를 소수로 쓰면?

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=____, random_state=0)

print('훈련용 :', x_train.shape)
print('시험지 :', x_test.shape)

훈련용 **200줄**, 시험지 **50줄**. 250의 20% 가 50이지요.

> **왼쪽에 변수가 네 개** 붙은 것을 보세요.
> `train_test_split` 이 결과를 **네 개 한꺼번에** 돌려주고,
> 왼쪽 네 이름이 **순서대로** 받습니다. 순서를 바꿔 쓰면 엉뚱한 것이 들어갑니다.

---
## ⑤ 모델 만들고 학습시키기

**이 단원에서 가장 중요한 두 줄**입니다.

> **힌트** · 학습시키는 명령은 «(몸에) 맞추다»라는 뜻의 세 글자입니다.

In [ ]:
model = LinearRegression()

model.____(x_train, y_train)

끝입니다. 이 한 줄 안에서 **경사하강법이 수백 번** 돌며 가장 덜 틀리는 직선을 찾았습니다.
[수업 자료에서 손으로 움직여 본 그것](https://richee-pc.github.io/AI_cs/unit2.html#algo)을 컴퓨터가 대신 한 겁니다.

찾아낸 직선을 꺼내 봅시다. `coef_` 가 기울기, `intercept_` 가 절편입니다.

In [ ]:
model.coef_, model.intercept_

이렇게 읽습니다.

```
청년 = 14242 × 도서관 + 32 × 공연장 - 9830
```

**도서관이 하나 늘 때 청년이 14,242명 는다**고 모델은 본 것입니다.
공연장은 32명 — **도서관이 훨씬 크게 작용**한다고 판단했네요.

> ⚠ **«도서관을 지으면 청년이 온다»는 뜻이 아닙니다.**
> 같이 움직인다는 것뿐이지 **원인이라는 증거는 아닙니다.**
> 청년이 많은 동네라서 도서관을 지었을 수도 있으니까요.

---
## ⑥ 맞혀 보고 점수 내기

> **힌트** · 맞혀 보라는 명령은 «예측하다»라는 영어 낱말입니다.

In [ ]:
Predicted = model.____(x_test)

Predicted[:5]     # 모델이 맞힌 값 다섯 개

`[:5]` 는 «앞에서 다섯 개만»이라는 뜻입니다(**슬라이싱**).

진짜 정답과 나란히 놓고 봅시다.

In [ ]:
y_test.head()

제법 비슷한가요? 눈대중 말고 **숫자로** 재 봅시다.

In [ ]:
print('평균제곱오차(MSE) :', mean_squared_error(y_test, Predicted))
print('결정계수(R²)      :', model.score(x_test, y_test))

**MSE 는 작을수록, R² 는 1에 가까울수록** 좋습니다.

교과서 107쪽에는 **MSE 약 9억**, **R² 약 0.64** 가 나옵니다. 여러분 화면도 비슷할 겁니다.

> **9억이면 엄청 나쁜 것 아닌가요?**
> 아닙니다. MSE 는 **오차를 제곱**한 값이라 단위가 «명²» 입니다.
> 청년 인구가 몇만 명 단위니까, 제곱하면 저렇게 커집니다.
> 그래서 MSE 는 **다른 모델과 견줄 때** 쓰고, 혼자서는 큰 의미가 없습니다.
>
> **R² = 0.64** 는 «평균만 찍는 것보다 꽤 낫다»는 뜻입니다.
> 1이면 완벽, 0이면 평균 찍는 것과 같습니다.

### 한 걸음 더

`x = df[['도서관', '공연장']]` 에 **박물관과 미술관도 넣어** 다시 돌려 보세요.
R² 가 올라가나요? **속성을 더 넣으면 늘 좋아질까요?**

---
# 뒷부분 · 분류 모델

## 붓꽃 세 종류를 가려내 봅시다

이번에는 **숫자가 아니라 «무엇인지»** 를 맞힙니다.
꽃잎과 꽃받침의 길이·너비를 보고 **setosa · versicolor · virginica** 중 무엇인지 가려냅니다.

쓰는 알고리즘은 **k-최근접 이웃** — [수업 자료에서 E를 끌어 옮겨 본](https://richee-pc.github.io/AI_cs/unit2.html#algo) 그것입니다.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import sklearn.metrics

df = pd.read_csv(주소 + 'iris.csv')
df.head()

## ⚠ 잠깐 — 열 이름부터 확인하고 갑니다

**교과서 코드를 그대로 치면 오류가 납니다.** 왜 그런지 직접 봅시다.

In [ ]:
list(df.columns)

보이시나요? 실제 파일의 열 이름에는 **띄어쓰기가 들어 있습니다** — `'꽃잎 길이'`.
그런데 교과서 109쪽은 `'꽃잎길이'` 라고 붙여 썼습니다.

붙여 쓴 채로 돌리면 이렇게 됩니다. **일부러 한 번 내 보세요.**

In [ ]:
df[['꽃잎길이', '꽃잎너비']]

`KeyError: "None of [...] are in the [columns]"`
→ **«그런 이름의 열은 없다»** 는 뜻입니다.

띄어쓰기 하나 때문입니다. 그래서 **데이터를 읽으면 `df.columns` 부터 찍어 보는 습관**이 중요합니다.
남이 만든 데이터는 **내 생각과 다르게 생겼을 수 있습니다.**

> 이것이 오늘 배우는 것 중 실전에서 제일 자주 만나는 문제입니다.
> 오타·띄어쓰기·대소문자 — 셋 다 컴퓨터에게는 **완전히 다른 이름**입니다.

---
## ③④ 나누기 — 앞부분과 똑같습니다

> **힌트** · 위에서 확인한 **진짜 열 이름**을 그대로 적으세요. 띄어쓰기까지요.

In [ ]:
x = df[['____', '____', '____', '____']]
y = df['종']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=10)

x_train.shape, x_test.shape

훈련용 **120줄**, 시험지 **30줄**.

`y` 는 대괄호 **한 겹**입니다. 정답이 한 열뿐이라 그렇습니다 — 두 겹으로 써도 돌아갑니다.

---
## ⑤ 모델 만들고 학습 — **여기 한 줄만 바뀝니다**

앞부분에서는 `LinearRegression()` 이었지요.
이번에는 `KNeighborsClassifier()` 입니다. **그것 말고는 글자 하나 안 바뀝니다.**

> **힌트** · 학습시키는 명령은 앞부분과 똑같은 세 글자입니다.

In [ ]:
model = KNeighborsClassifier()

model.____(x_train, y_train)

> `KNeighborsClassifier()` 괄호 안이 비었지요? **k 를 안 정해 주면 기본값 5** 를 씁니다.
> `KNeighborsClassifier(n_neighbors=3)` 처럼 바꿔 볼 수 있습니다. **k 는 사람이 정하는 값**(하이퍼파라미터)이니까요.

---
## ⑥ 맞혀 보기

In [ ]:
Predicted = model.predict(x_test)

print('모델의 답 :', Predicted[:10])
print('진짜 정답 :', y_test.values[:10])

두 줄을 **하나씩 견주어** 보세요. 다른 자리가 보이나요?

교과서 110쪽에서는 **일곱 번째**가 틀렸습니다 — 모델은 2(virginica)라 했는데 정답은 1(versicolor)이었지요.

30개 전부를 한눈에 보려면 **오분류표**를 씁니다.

In [ ]:
sklearn.metrics.confusion_matrix(y_test, Predicted)

이렇게 읽습니다.

```
          실제 0   실제 1   실제 2
 예측 0  [  10       0       0  ]   ← setosa 는 열 개 다 맞혔다
 예측 1  [   0      12       1  ]   ← virginica 하나를 versicolor 라고 잘못 봤다
 예측 2  [   0       0       7  ]
```

**대각선이 맞힌 것**입니다. 대각선을 벗어난 숫자가 틀린 개수예요.

> 오분류표를 읽을 줄 알면 **정확도·정밀도·재현율이 전부 여기서 나옵니다.**
> 시험에 표를 주고 계산하게 하는 문제가 나옵니다.

---
## 점수 매기기

In [ ]:
print('정확도 :', sklearn.metrics.accuracy_score(y_test, Predicted))
print('정밀도 :', sklearn.metrics.precision_score(y_test, Predicted, average=None))
print('재현율 :', sklearn.metrics.recall_score(y_test, Predicted, average=None))

**정확도 약 0.967** — 30개 중 29개를 맞혔습니다.

$$정확도 = \frac{맞힌\ 개수}{전체} = \frac{10+12+7}{30} = 0.9666...$$

**정밀도**와 **재현율**은 종류마다 따로 나옵니다.

| | 묻는 것 | 언제 중요한가 |
|---|---|---|
| **정밀도** | 「2번이라고 한 것 중 **진짜 2번**은?」 | **스팸 분류** — 아닌 걸 스팸함에 넣으면 못 본다 |
| **재현율** | 「진짜 2번 중 **찾아낸 것**은?」 | **암 진단** — 하나라도 놓치면 안 된다 |

정확도 하나만 보면 속을 때가 있습니다.
1,000명 중 환자가 1명인 병에서 **«전원 정상»이라고만 답해도 정확도는 99.9%** 입니다.
그런데 그 모델은 **환자를 한 명도 못 찾았습니다.** 재현율은 0이지요.

---
## 다 했습니다 · 확인해 보세요

- [ ] 모델 만드는 여섯 단계를 순서대로 말할 수 있다
- [ ] **훈련 데이터**는 공부용, **테스트 데이터**는 시험지
- [ ] `fit()` 은 **학습**, `predict()` 는 **맞혀 보기**
- [ ] 예측 모델은 **MSE·R²**, 분류 모델은 **정확도·정밀도·재현율**
- [ ] 예측과 분류는 **모델 만드는 한 줄만** 다르다
- [ ] 데이터를 읽으면 **`df.columns` 부터** 확인한다

### 한 걸음 더 (시간이 남으면)

`KNeighborsClassifier(n_neighbors=1)` 로 바꿔 정확도를 다시 보세요.
그다음 `n_neighbors=15`. **k 를 키우면 늘 좋아지나요?**
[수업 자료에서 E를 끌어 옮겼던 그 이야기](https://richee-pc.github.io/AI_cs/unit2.html#algo)와 이어집니다.

막히면 👉 [에러 응급처치 사전](https://richee-pc.github.io/AI_cs/colab.html)

*made by ptp 🐰*